# Synthetic A/A-test with a Monte Carlo approach

### Importing the packages

In [1]:
import pandahouse as ph
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import ttest_ind
from tqdm import tqdm # для отслеживания прогресса при симуляции данных и рассчёте t-test


In [2]:
rng = np.random.default_rng() # for a more optimized sample generation (less memory use)

Building the environment and access to the database

In [3]:
from dotenv import load_dotenv
import os
load_dotenv()

connection = {
    'host': os.getenv('CLICKHOUSE_HOST'),
    'database': os.getenv('CLICKHOUSE_DATABASE'),
    'user': os.getenv('CLICKHOUSE_USER'),
    'password': os.getenv('CLICKHOUSE_PASSWORD')
}

## Conducting the synthetic A/A-test for a planned 1-week period experiment

### Querying the data on views

In [11]:
q = '''
SELECT 
    views, 
    count() as users
FROM
(
SELECT 
    user_id, 
    sum(action = 'view') as views
FROM {db}.feed_actions
WHERE time::date BETWEEN '2025-11-14' AND '2025-11-20'
GROUP BY user_id
)
GROUP BY views
'''.format(db=connection['database'])
views = ph.read_clickhouse(q, connection = connection)
views['p'] = views.users / views.users.sum() # the probability for each user to be in the further simulation sample

Checking th dataframe for the correct download

In [12]:
views.head()

,views,users,p
0,198,25,0.000595
1,116,159,0.003786
2,66,336,0.008001
3,272,2,0.000048
4,240,7,0.000167


Calculating the overall sample for the analysis

In [13]:
print(f"The overall sample size: {views.users.sum():.0f}")

The overall sample size: 41997


Further, calculating the subsample size for two groups for the 50/50 split

In [14]:
print(f"The sample size in each group: {views.users.sum()//2:.0f}")

The sample size in each group: 20998


### Querying and calculating the CTR data

In [20]:
q = """
SELECT 
   floor(ctr, 2) as ctr, 
   count() as users
FROM (SELECT toDate(time) as dt, 
    user_id,
    sum(action = 'like')/sum(action = 'view') as ctr
FROM {db}.feed_actions
WHERE dt between '2025-11-14' and '2025-11-20'
GROUP BY dt, user_id
)
GROUP BY ctr
""".format(db=connection['database'])

ctr = ph.read_clickhouse(q, connection = connection)
ctr['p'] = ctr.users / ctr.users.sum() # calculating the probability for each user to get into the simulation

Checking the correctness of the dataframe download

In [17]:
ctr.head()

,ctr,users,p
0,0.00,1443,0.016952
1,0.65,4,0.000047
2,0.71,5,0.000059
3,0.49,4,0.000047
4,0.54,72,0.000846


### Calculating the t-test and the test power

In [21]:
#ttest

beta = 0

sample_size = views.users.sum() // 2 # 20998


for i in tqdm(range(20000)):
    
    # let the group_A be a test group. Then, I create a sample for views for the test group as following:
    group_A_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # Changing the data with a correction to the algorythm effect (adjusted by the ML team formula) for the test group:
    group_A_views = group_A_views + ((1 + rng.binomial(1, 0.5, size = sample_size)) * rng.binomial(1, 0.9, size = sample_size)) * (group_A_views >= 50)
    
    
    # group_B - is a control group. Creating a subsample:
    group_B_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # creating a CTR sample for each group:
    group_A_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    group_B_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    
    # simulating the likes' distribution for each group:
    group_A_likes = rng.binomial(group_A_views, group_A_ctr)
    group_B_likes = rng.binomial(group_B_views, group_B_ctr)
    
    # calculating the Welch's t-test, appending the powers of statistically significant tests (p-value <= 0.05) to the 'beta' object:
    beta += (stats.ttest_ind(group_B_likes, group_A_likes, equal_var = False).pvalue <= 0.05)

# calculating the final overall power for the tests (in percentages):
result = (beta / 20000) * 100
print(f"The potential power of the planned test (in %): {result:.4f}")

100%|██████████| 20000/20000 [07:54<00:00, 42.12it/s]   

The potential power of the planned test (in %): 24.8300


So, the test power is less then 30%. This means that with the given parameters of the algorythm and a 1-week period of the potential experiment, we will 'catch' the effect from the new algorythm (if any) only in 30% of such cases.

### What if the new algorythm has an effect with 30+ views?

Later, the ML team improved the algorythm, and now it works for users with 30+ views of the feed. So, now I adjust the calculations of the test power accounting for this correction:

In [22]:
#ttest

beta = 0

for i in tqdm(range(20000)):
    
    # group_A is a test group. Then, the views' sample for this group is:
    group_A_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # changing the data with a correction of the algorythm effect for the test group:
    group_A_views = group_A_views + ((1 + rng.binomial(1, 0.5, size = sample_size)) * rng.binomial(1, 0.9, size = sample_size)) * (group_A_views >= 30)
    
    
    # group_B is a control group. Creating the sample for it:
    group_B_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # creating the CTR sample for each group:
    group_A_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    group_B_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    
    # simulating the likes' distribution for each group:
    group_A_likes = rng.binomial(group_A_views, group_A_ctr)
    group_B_likes = rng.binomial(group_B_views, group_B_ctr)
    
    # calculating the Welch's t-test, appending the powers of the statistically significant results (p-value <= 0.05) to beta:
    beta += (stats.ttest_ind(group_B_likes, group_A_likes, equal_var = False).pvalue <= 0.05)

# calculating the final overall power of the tests:
result = (beta / 20000) * 100
print(f"The potential power of the planned test (in %): {result:.4f}")

100%|██████████| 20000/20000 [05:13<00:00, 63.86it/s]

The potential power of the planned test (in %): 41.6700


We can see that, indeed, the test power is higher. Now, we can find the effect in ~41% of cases where the effect is present.

## What if the test lasts for 2 weeks?

According to the new terms from the business, the potential test will last for 2 weeks. Now, I need to calculate the test power for this period length.

Also, we assume that within this 2-week period, we will have approximately the same number of users as we had in total for the A/A-test period. So, I will take this assumption into account when calculating the sample for the A/A-test.

### Querying the data on views

In [23]:
q = '''
SELECT 
    views, 
    count() as users
FROM
(
SELECT 
    user_id, 
    sum(action = 'view') as views
FROM {db}.feed_actions
WHERE time::date BETWEEN '2025-11-14' AND '2025-11-20'
GROUP BY user_id
)
GROUP BY views
'''.format(db=connection['database'])

views = ph.read_clickhouse(q, connection = connection)
views['p'] = views.users / views.users.sum()

Checking the data correctness:

In [24]:
views.sort_values(by = 'p', ascending = True)

,views,users,p
238,316,1,0.000024
280,301,1,0.000024
139,287,1,0.000024
137,255,1,0.000024
30,271,1,0.000024
...,...,...,...
16,30,469,0.011167
265,35,485,0.011548
116,14,500,0.011906
49,15,537,0.012787


### Querying the CTR data

In [25]:
q = """
select 
   floor(ctr, 2) as ctr, 
   count() as users
from (select toDate(time) as dt, 
    user_id,
    sum(action = 'like')/sum(action = 'view') as ctr
from {db}.feed_actions
where dt between '2025-11-14' and '2025-11-20'
group by dt, user_id
)
group by ctr
""".format(db=connection['database'])

ctr = ph.read_clickhouse(q, connection = connection)
ctr['p'] = ctr.users / ctr.users.sum()

Checking its correctness:

In [26]:
ctr.sort_values(by = 'ctr', ascending = True)

,ctr,users,p
0,0.00,1443,0.016952
69,0.02,48,0.000564
23,0.03,142,0.001668
32,0.04,312,0.003665
16,0.05,727,0.008541
...,...,...,...
12,0.81,2,0.000023
73,0.83,1,0.000012
26,0.85,3,0.000035
39,0.88,1,0.000012


### Querying the data for a 2-week period to calculate the sample size according to the assumption

In [27]:
q = '''
SELECT 
    views, 
    count() as users
FROM
(
SELECT 
    user_id, 
    sum(action = 'view') as views
FROM {db}.feed_actions
WHERE time::date BETWEEN '2025-11-14' AND '2025-11-27'
GROUP BY user_id
)
GROUP BY views
'''.format(db=connection['database'])

views_2 = ph.read_clickhouse(q, connection = connection)

### Calculating the t-test and power

In [28]:
#ttest

sample_size = views_2.users.sum() // 2 # 30591 in each group

beta = 0

for i in tqdm(range(20000)):
    
    # group_A is a test group. So, the views' sample for this group is:
    group_A_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # changing the data with an algorythm correction for the test group:
    group_A_views = group_A_views + ((1 + rng.binomial(1, 0.5, size = sample_size)) * rng.binomial(1, 0.9, size = sample_size)) * (group_A_views >= 30)
    
    
    # group_B is a control group. The sample:
    group_B_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # creating a CTR sample for each group:
    group_A_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    group_B_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    
    # simulating the likes' distribution for each group:
    group_A_likes = rng.binomial(group_A_views, group_A_ctr)
    group_B_likes = rng.binomial(group_B_views, group_B_ctr)
    
    # calculating the Welch's t-test, appending the power of the significant test (p-value <= 0.05) to beta:
    beta += (stats.ttest_ind(group_B_likes, group_A_likes, equal_var = False).pvalue <= 0.05)

# calculating the final overall test power:
result = (beta / 20000) * 100
print(f"The potential power of the planned test (in %): {result:.4f}")

100%|██████████| 20000/20000 [07:27<00:00, 44.66it/s]

The potential power of the planned test (in %): 56.3050


With a larger time period of the test (and, consequently, a larger sample), the test power is even higher. Now, the significant effect (if any) will be found in ~56% of such cases.

## Calculating the test power only for users potentially affected by the algorythm

Until now, I analyzed the total samples, including both those users potentially affected by the algorythm (30+ views) and those with potentially no effect (<30 views). The team lead suggested to check the test power only for the affected users. The logic is the following: yes, the sample will be smaller, but this will help us to avoid the potential noise and, thus, the test sensitivity may be stronger, so we may find a clear effect for the potential audience.

The rest of the assumptions are the same: the test period is 2 weeks, and the algorythm has an effect on users with 30+ views of the news feed.

### Querying the views' data

In [29]:
q = '''
SELECT 
    views, 
    count() as users
FROM
(
SELECT 
    user_id, 
    sum(action = 'view') as views
FROM {db}.feed_actions
WHERE time::date BETWEEN '2025-11-14' AND '2025-11-20'
GROUP BY user_id
)
GROUP BY views
'''.format(db=connection['database'])

views = ph.read_clickhouse(q, connection = connection)
views['p'] = views.users / views.users.sum()

### Querying the CTR data

In [30]:
q = """
select 
   floor(ctr, 2) as ctr, 
   count() as users
from (select toDate(time) as dt, 
    user_id,
    sum(action = 'like')/sum(action = 'view') as ctr
from {db}.feed_actions
where dt between '2025-11-14' and '2025-11-20'
group by dt, user_id
)
group by ctr
""".format(db=connection['database'])

ctr = ph.read_clickhouse(q, connection = connection)
ctr['p'] = ctr.users / ctr.users.sum()

### Calculating the t-test and power

In [31]:
#ttest

sample_size = views_2.users.sum() // 2 # 30591

beta = 0

for i in tqdm(range(20000)):
    
    # group_A is a test group. The views' sample is:
    group_A_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # changing the data with a correction of the ML algorythm formula for the test group:
    group_A_views = group_A_views + ((1 + rng.binomial(1, 0.5, size = sample_size)) * rng.binomial(1, 0.9, size = sample_size)) * (group_A_views >= 30)
    
    
    # group_B is a control group, the sample is:
    group_B_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # creating a CTR sample for each group:
    group_A_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    group_B_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    
    # simulating a likes' distribution for each group:
    group_A_likes = rng.binomial(group_A_views, group_A_ctr)
    group_B_likes = rng.binomial(group_B_views, group_B_ctr)
    
    # creating the filters for the final test only for users with 30+ views:
    mask_A = group_A_views >= 30
    mask_B = group_B_views >= 30
    
    # calculating the Welch's t-test, appending the results (p-value <= 0.05) to beta:
    beta += (stats.ttest_ind(group_B_likes[mask_B], group_A_likes[mask_A], equal_var = False).pvalue <= 0.05)

# calculating the final power:
result = (beta / 20000) * 100
print(f"The potential power of the planned test (in %): {result:.4f}")


100%|██████████| 20000/20000 [07:37<00:00, 43.72it/s]

The potential power of the planned test (in %): 63.7600


As a result, we get a test power of ~64%. So, even if we analyze the algorythm effect for the selected users with 30+ views (potential effect), the test power, though increasing, is still not high enough: we may find the effect only in 64% of cases, which is lower than the conventional benchmark of 80%.

# Recommendations

Based on the analysis results, I would recommend **not to launch** the experiment and the new algorythm until further improvements. The possible further steps may include:
* increase the time period of the experiment and enlarge the sample
* to improve the ML-model (adjust the mechanism to affect users with <30 views): it seems, the current version of the algorythm implies a too small effect for the metrics
